<a href="https://colab.research.google.com/github/anyuanay/info212/blob/main/INFO212_Week6_Lecture_Data_Loading_concat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 212: Data Science Programming 1
___

### Data Loading, Storage, and File Formats
---

**Question:**
- How to interact with external data sources using Python? 

**Objectives:**
- Read and write data from text formats
- Read and write data using different delimiters
- Read and write JSON data
- Read and write HTML and XML data
- Scrape data from the Web
- Read and write data in binary formats

## Course Project: Teams and Proposal 
Any questions about project teams and proposal?

# Loading Data
## Accessing data is a necessary first step for data analysis. We are going to be focused on data input and output using pandas, though there are numerous tools in other libraries to help with reading and writing data in various formats. Input and output typically falls into a few main categories: reading text files and other more efficient on-disk formats, loading data from databases, and interacting with network sources like web APIs.

In [ ]:
# import 


# XML and HTML: Web Scraping
### Python has many libraries for reading and writing data in the ubiquitous HTML and XML formats. Examples include lxml, Beautiful Soup, and html5lib. While lxml is comparatively much faster in general, the other libraries can better handle malformed HTML or XML files.

### Pandas has a built-in function, read_html, which uses libraries like lxml and Beautiful Soup to automatically parse tables out of HTML files as DataFrame objects. To show how this works, I downloaded an HTML file (used in the pandas documentation) from the United States FDIC government agency showing bank failuresm (https://www.fdic.gov/bank/individual/failed/banklist.html). First, you must install some additional libraries used by read_html:

In [ ]:
!pip install lxml beautifulsoup4 html5lib

In [ ]:
from google.colab import files
files.upload()

In [ ]:
tables = pd.read_html('fdic_failed_bank_list.html')

In [ ]:
type(tables)

In [ ]:
len(tables)

In [ ]:
tables.head()

In [ ]:
type(tables)

In [ ]:
len(tables)

In [ ]:
df = tables[0]
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
close_timestamps = pd.to_datetime(failures['Closing Date'])

In [ ]:
type(close_timestamps[0])

In [ ]:
close_timestamps.dt.year.value_counts()

In [ ]:
close_timestamps.dt.year.value_counts().sort_index()

# JSON Data
## JSON (short for JavaScript Object Notation) has become one of the standard formats for sending data by HTTP request between web browsers and other applications. It is a much more free-form data format than a tabular text form like CSV. Here is an example.

In [ ]:
obj = """
{"name": "Wes",
 "places_lived": ["United States", "Spain", "Germany"],
 "pet": null,
 "siblings": [{"name": "Scott", "age": 30, "pets": ["Zeus", "Zuko"]},
              {"name": "Katie", "age": 38,
               "pets": ["Sixes", "Stache", "Cisco"]}]
}
"""

## How to parse a JSON structure into a Python object?

In [ ]:
import json
result = json.loads(obj)
type(result)

In [ ]:
import json

In [ ]:
result = json.loads(obj)

In [ ]:
result

In [ ]:
type(result)

## How to convert a Python object back to JSON?

In [ ]:
asjson = json.dumps(result)
asjson

In [ ]:
type(asjson)

In [ ]:
type(result)

## How you convert a JSON object or list of objects to a DataFrame or some other data structure for analysis will be up to you. Conveniently, you can pass a list of dicts (which were previously JSON objects) to the DataFrame constructor and select a subset of the data fields.

In [ ]:
result

In [ ]:
result_df = pd.DataFrame(result['siblings'])

In [ ]:
result_df.head()

In [ ]:
siblings = pd.DataFrame(result['siblings'], columns=['name', 'age'])
siblings

In [ ]:
ex = {"a":[1, 2, 3, 4 ], "b": [3, 4,5], 'c': [5, 6, 7]}

In [ ]:
pd.DataFrame(ex)

## The pandas `read_json` can automatically convert JSON datasets in specific arrangements into a Series or DataFrame.

In [ ]:
from google.colab import files
files.upload()

In [ ]:
data = pd.read_json('example.json')
data

In [ ]:
data

In [ ]:
data.to_json()

In [ ]:
data.to_json(orient='records')

In [ ]:
print(data.to_json())
print(data.to_json(orient='records'))

In [ ]:
type(data.to_json())

# Binary Data Formats

## One of the easiest ways to store data (also known as serialization) efficiently in binary format is using Python’s built-in pickle serialization. pandas objects all have a to_pickle method that writes the data to disk in pickle format:

In [ ]:
frame = pd.read_csv('examples/ex1.csv')
frame

In [ ]:
frame.to_pickle('examples/frame_pickle')

In [ ]:
pd.read_pickle('examples/frame_pickle')

# Reading Microsoft Excel Files

## pandas also supports reading tabular data stored in Excel 2003 (and higher) files using either the ExcelFile class or pandas.read_excel function. Internally these tools use the add-on packages xlrd and openpyxl to read XLS and XLSX files, respectively.

## To use ExcelFile, create an instance by passing a path to an xls or xlsx file:

In [ ]:
files.upload()

In [ ]:
xlsx = pd.ExcelFile('ex1.xlsx')

In [ ]:
xlsx

## Data stored in a sheet can then be read into DataFrame with parse:

In [ ]:
pd.read_excel(xlsx, 'Sheet1')

In [ ]:
pd.read_excel(xlsx, 'Sheet1', index_col=0)

## If you are reading multiple sheets in a file, then it is faster to create the ExcelFile, but you can also simply pass the filename to pandas.read_excel:

In [ ]:
frame = pd.read_excel('ex1.xlsx', 'Sheet1', index_col=0)
frame

In [ ]:
frame =pd.read_excel('examples/ex1.xlsx', "Sheet1", index_col=0)

In [ ]:
frame

## To write pandas data to Excel format, you must first create an ExcelWriter, then write data to it using pandas objects’ to_excel method:

In [ ]:
writer = pd.ExcelWriter('examples/ex2.xlsx')
frame.to_excel(writer, 'Sheet1')
writer.save()

## You can also pass a file path to to_excel and avoid the ExcelWriter:

In [ ]:
frame.to_excel('examples/ex2.xlsx')

# Combining and Merging Datasets

## Database-Style DataFrame Joins
## Merge or join operations combine datasets by linking rows using one or more keys. These operations are central to relational databases (e.g., SQL-based). The merge function in pandas is the main entry point for using these algorithms on your data. Let’s start with a simple example:

In [ ]:
df1 = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'a', 'b'],
                    'data1': range(7)})
df2 = pd.DataFrame({'key': ['a', 'b', 'd'],
                    'data2': range(3)})
df1

In [ ]:
df2

In [ ]:
df2

## This is an example of a many-to-one join; the data in df1 has multiple rows labeled a and b, whereas df2 has only one row for each value in the key column. Calling merge with these objects we obtain:

In [ ]:
pd.merge(df1, df2)

## Note that I didn’t specify whichcolumn to join on. If that information is not specified, merge uses the overlapping column names as the keys. It’s a good practice to specify explicitly, though:

In [ ]:
pd.merge(df1, df2, left_on='key')

## If the column names are different in each object, you can specify them separately:

In [ ]:
df3 = pd.DataFrame({'lkey': ['b', 'b', 'a', 'c', 'a', 'a', 'b'],
                    'data1': range(7)})
df4 = pd.DataFrame({'rkey': ['a', 'b', 'd'],
                    'data2': range(3)})

In [ ]:
df3

In [ ]:
df4

In [ ]:
pd.merge(df3, df4, left_on='lkey', right_on='rkey')

## You may notice that the 'c' and 'd' values and associated data are missing from the result. By default merge does an 'inner' join; the keys in the result are the intersection, or the common set found in both tables. Other possible options are 'left', 'right', and 'outer'. The outer join takes the union of the keys, combining the effect of applying both left and right joins:

In [ ]:
pd.merge(df1, df2, how='outer')

## Many-to-many merges have well-defined, though not necessarily intuitive, behavior. Here’s an example:

In [ ]:
df1 = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'b'],
                    'data1': range(6)})
df2 = pd.DataFrame({'key': ['a', 'b', 'a', 'b', 'd'],
                    'data2': range(5)})
df1
df2
pd.merge(df1, df2, on='key', how='left')

## Many-to-many joins form the Cartesian product of the rows. Since there were three 'b' rows in the left DataFrame and two in the right one, there are six 'b' rows in the result. The join method only affects the distinct key values appearing in the result:

In [ ]:
pd.merge(df1, df2, how='inner')

## To merge with multiple keys, pass a list of column names:

In [ ]:
left = pd.DataFrame({'key1': ['foo', 'foo', 'bar'],
                     'key2': ['one', 'two', 'one'],
                     'lval': [1, 2, 3]})
right = pd.DataFrame({'key1': ['foo', 'foo', 'bar', 'bar'],
                      'key2': ['one', 'one', 'one', 'two'],
                      'rval': [4, 5, 6, 7]})
pd.merge(left, right, on=['key1', 'key2'], how='outer')

## A last issue to consider in merge operations is the treatment of overlapping column names. While you can address the overlap manually (see the earlier section on renaming axis labels), merge has a suffixes option for specifying strings to append to overlapping names in the left and right DataFrame objects:

In [ ]:
pd.merge(left, right, on='key1')

In [ ]:
pd.merge(left, right, on='key1', suffixes=('_left', '_right'))

## Merging on Index
## In some cases, the merge key(s) in a DataFrame will be found in its index. In this case, you can pass left_index=True or right_index=True (or both) to indicate that the index should be used as the merge key:

In [ ]:
left1 = pd.DataFrame({'key': ['a', 'b', 'a', 'a', 'b', 'c'],
                      'value': range(6)})
right1 = pd.DataFrame({'group_val': [3.5, 7]}, index=['a', 'b'])
left1

In [ ]:
right1

In [ ]:
pd.merge(left1, right1, left_on='key', right_index=True)

## Since the default merge method is to intersect the join keys, you can instead form the union of them with an outer join:

In [ ]:
pd.merge(left1, right1, left_on='key', right_index=True, how='outer')

## With hierarchically indexed data, things are more complicated, as joining on index is implicitly a multiple-key merge:

In [ ]:
lefth = pd.DataFrame({'key1': ['Ohio', 'Ohio', 'Ohio',
                               'Nevada', 'Nevada'],
                      'key2': [2000, 2001, 2002, 2001, 2002],
                      'data': np.arange(5.)})
righth = pd.DataFrame(np.arange(12).reshape((6, 2)),
                      index=[['Nevada', 'Nevada', 'Ohio', 'Ohio',
                              'Ohio', 'Ohio'],
                             [2001, 2000, 2000, 2000, 2001, 2002]],
                      columns=['event1', 'event2'])
lefth

In [ ]:
righth

## In this case, you have to indicate multiple columns to merge on as a list (note the handling of duplicate index values with how='outer'):

In [ ]:
pd.merge(lefth, righth, left_on=['key1', 'key2'], right_index=True)

In [ ]:
pd.merge(lefth, righth, left_on=['key1', 'key2'],
         right_index=True, how='outer')

## Using the indexes of both sides of the merge is also possible:

In [ ]:
left2 = pd.DataFrame([[1., 2.], [3., 4.], [5., 6.]],
                     index=['a', 'c', 'e'],
                     columns=['Ohio', 'Nevada'])
right2 = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [13, 14]],
                      index=['b', 'c', 'd', 'e'],
                      columns=['Missouri', 'Alabama'])
left2

In [ ]:
right2

In [ ]:
pd.merge(left2, right2, how='outer', left_index=True, right_index=True)

## DataFrame has a convenient join instance for merging by index. It can also be used to combine together many DataFrame objects having the same or similar indexes but non-overlapping columns. In the prior example, we could have written:

In [ ]:
left2.join(right2, how='outer')

## In part for legacy reasons (i.e., much earlier versions of pandas), DataFrame’s join method performs a left join on the join keys, exactly preserving the left frame’s row index. It also supports joining the index of the passed DataFrame on one of the columns of the calling DataFrame:

In [ ]:
left1.join(right1, on='key')

## Lastly, for simple index-on-index merges, you can pass a list of DataFrames to join as an alternative to using the more general concat function described in the next section:

In [ ]:
another = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [16., 17.]],
                       index=['a', 'c', 'e', 'f'],
                       columns=['New York', 'Oregon'])
another

In [ ]:
left2.join([right2, another])

In [ ]:

left2.join([right2, another], how='outer')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## Concatenating Along an Axis
## Another kind of data combination operation is referred to interchangeably as concatenation,binding, or stacking. NumPy’s concatenate function can do this with NumPy arrays:

In [ ]:
arr = np.arange(12).reshape((3, 4))
arr

In [ ]:
np.concatenate([arr, arr], axis =1)

In [ ]:
np.concatenate([arr, arr], axis=1)

In [ ]:
np.concatenate([arr, arr])

## In the context of pandas objects such as Series and DataFrame, having labeled axes enable you to further generalize array concatenation. In particular, you have a number of additional things to think about:
* If the objects are indexed differently on the other axes, should we combine the
distinct elements in these axes or use only the shared values (the intersection)?
* Do the concatenated chunks of data need to be identifiable in the resulting
object?
* Does the “concatenation axis” contain data that needs to be preserved? In many
cases, the default integer labels in a DataFrame are best discarded during
concatenation.

## The concat function in pandas provides a consistent way to address each of these concerns. Here is a list of examples to illustrate how it works. Suppose we have three Series with no index overlap:

In [ ]:
s1 = pd.Series([0, 1], index=['a', 'b'])
s2 = pd.Series([2, 3, 4], index=['c', 'd', 'e'])
s3 = pd.Series([5, 6], index=['f', 'g'])

In [ ]:
s1

## Calling concat with these objects in a list glues together the values and indexes:

In [ ]:
pd.concat([s1, s2, s3], axis=1)

## By default concat works along axis=0, producing another Series. If you pass axis=1, the result will instead be a DataFrame (axis=1 is the columns):

In [ ]:
pd.concat([s1, s2, s3], axis=1)

## In this case there is no overlap on the other axis, which as you can see is the sorted union (the 'outer' join) of the indexes. You can instead intersect them by passing join='inner':

In [ ]:
s4 = pd.concat([s1, s3])
s4

In [ ]:
pd.concat([s1, s4], axis=1)

In [ ]:
pd.concat([s1, s4], axis=1, join='inner')

## In this last example, the 'f' and 'g' labels disappeared because of the join='inner' option.

## A potential issue is that the concatenated pieces are not identifiable in the result. Suppose instead you wanted to create a hierarchical index on the concatenation axis. To do this, use the keys argument:

In [ ]:
result = pd.concat([s1, s1, s3], keys=['one', 'two', 'three'])
result

In [ ]:
result.unstack()

## In the case of combining Series along axis=1, the keys become the DataFrame column headers:

In [ ]:
pd.concat([s1, s2, s3], axis=1, keys=['one', 'two', 'three'])

## The same logic extends to DataFrame objects:

In [ ]:
df1 = pd.DataFrame(np.arange(6).reshape(3, 2), index=['a', 'b', 'c'],
                   columns=['one', 'two'])
df2 = pd.DataFrame(5 + np.arange(4).reshape(2, 2), index=['a', 'c'],
                   columns=['three', 'four'])
df1

In [ ]:
df2

In [ ]:
pd.concat([df1, df2])

In [ ]:
pd.concat([df1, df2], axis=1)

In [ ]:
pd.concat([df1, df2], axis=1, keys=['level1', 'level2'])

## If you pass a dict of objects instead of a list, the dict’s keys will be used for the keys option:

In [ ]:
pd.concat({'level1': df1, 'level2': df2}, axis=1)

In [ ]:
pd.concat({'level1':df1, 'level2':df2}, axis=1)

## There are additional arguments governing how the hierarchical index is created. For example, we can name the created axis levels with the names argument:

In [ ]:
pd.concat([df1, df2], axis=1, keys=['level1', 'level2'], names=['df', 'feature'])

## A last consideration concerns DataFrames in which the row index does not contain any relevant data:

In [ ]:
df1 = pd.DataFrame(np.random.randn(3, 4), columns=['a', 'b', 'c', 'd'])
df2 = pd.DataFrame(np.random.randn(2, 3), columns=['b', 'd', 'a'])
df1

In [ ]:
df2

In [ ]:
pd.concat([df1, df2], ignore_index=False)

## In this case, you can pass ignore_index=True:

In [ ]:
pd.concat([df1, df2], ignore_index=True)

# Reading and Writing Data in Text Format
## pandas features a number of functions for reading tabular data as a DataFrame object. The methods `read_csv` and `read_table` are likely the ones we’ll use the most.

In [ ]:
df = pd.read_csv('examples/ex1.csv', index_col=0)
df

## How to use `read_table()` to read a csv file?

In [ ]:
pd.read_table('examples/ex1.csv', sep=',')

## How to read a csv file without a header row?

## Let the Pandas to assign default header:

In [ ]:
pd.read_csv('examples/ex2.csv', header=None)

## Or specify your own header:

In [ ]:
names = ['a', 'b', 'c', 'd', 'message']
pd.read_csv('examples/ex2.csv', names=names)

## How to define a particular column as the index?

In [ ]:
pd.read_csv('examples/ex2.csv', names=names, index_col='message')

## How to use multiple columns for index?

In [ ]:
df = pd.read_csv('examples/ex2.csv', names=names, index_col=['message', 'a'])

In [ ]:
df.index

In [ ]:
df

## How to read a text file with irregular delimiters?
- Answer: we can pass in a regular expression for the keyword 'sep'.

In [ ]:
list(open('examples/ex3.txt'))

In [ ]:
result = pd.read_table('examples/ex3.txt', sep='\s+')
result

## The parser functions have many additional arguments to help you handle the wide variety of exception file formats that occur. For example, you can skip the first, third, and fourth rows of a file with skiprows:

In [ ]:
pd.read_csv('examples/ex4.csv', skiprows=[0, 2, 3])

## How to handle missing data in the input file?

In [ ]:
result = pd.read_csv('examples/ex5.csv')
result

In [ ]:
pd.isnull(result)

## You can specify `na_values` for treating missing value:

In [ ]:
result = pd.read_csv('examples/ex5.csv', na_values=[1])
result

## Different NA sentinels can be specified for each column in a dict:

In [ ]:
sentinels = {'message': ['foo', 'NA'], 'something': ['two']}
pd.read_csv('examples/ex5.csv', na_values=sentinels)

## Reading Text Files in Pieces
## How to read a certain number of lines from a file?

In [ ]:
pd.read_csv('examples/ex6.csv', nrows=10)

## Working with Delimited Formats
## It’s possible to load most forms of tabular data from disk using functions like pandas `read_table`. In some cases, however, some manual processing may be necessary. It’s not uncommon to receive a file with one or more malformed lines that trip up read_table. To illustrate the basic tools, consider a small CSV file:

In [ ]:
pd.read_csv("examples/ex7.csv")

## For any file with a single-character delimiter, you can use Python’s built-in csv module. To use it, pass any open file or file-like object to csv.reader:

In [ ]:
import csv
f = open('examples/ex7.csv')

reader = csv.reader(f)

In [ ]:
for line in reader:
    print(line)

## From there, it’s up to you to do the wrangling necessary to put the data in the form that you need it.

In [ ]:
with open('examples/ex7.csv') as f:
    lines = list(csv.reader(f))

In [ ]:
header, values = lines[0], lines[1:]

In [ ]:
data_dict = {h: v for h, v in zip(header, zip(*values))}
data_dict

## Writing Data to Text Format
## Data can also be exported to a delimited format.

In [ ]:
data = pd.read_csv('examples/ex5.csv')
data

In [ ]:
data.to_csv('examples/out.csv', sep='|', na_rep='NULL')

## How to write csv file using different deliminter?

In [ ]:
import sys
data.to_csv(sys.stdout, sep='|', na_rep='NULL')

## How to write null values explicitly?

In [ ]:
data.to_csv(sys.stdout, na_rep='NULL')

## How to ignore index and header labels in the written file?

In [ ]:
data.to_csv(sys.stdout, index=False, header=False)

## How to write out a subset of the columns?

In [ ]:
data.to_csv(sys.stdout, index=False, columns=['a', 'b', 'c'])